# Notebook 06c - Rendu HTML des solutions Z3 : visualisation structurée d'un planificateur de repas

**Navigation** : [Index Z3](./README.md) | [Index SymbolicAI](../../README.md) | [Index général](../../../README.md) | [Notebook 05 (Meal Planner)](./05_Meal_Planner_Hierarchical.ipynb) | [Notebook 06b (RecipeML)](./06b_RecipeML_Corpus.ipynb)

## Objectifs d'apprentissage

- Comprendre **pourquoi** le rendu d'une solution Z3 importe autant que sa résolution : un solveur n'a de valeur opérationnelle que si ses sorties sont **lisibles par un humain**
- Utiliser l'API `display(HTML(...))` de `Microsoft.DotNet.Interactive` pour produire des **cartes HTML** (tableaux colorés, badges, barres de progression) depuis un modèle C#
- Réutiliser les modèles du notebook 06b (sélection booléenne, plan `int[][]`) et **n'en changer que la couche de présentation** — le solveur reste identique
- Produire un **front de Pareto** rendu visuellement : le solveur explore l'espace des solutions sous plusieurs bornes de budget, et le résultat est rendu comme un tableau de compromis

## Prérequis

- Notebook [05 (Meal Planner Hierarchical)](./05_Meal_Planner_Hierarchical.ipynb) : pattern `int[][]`, `MkITE`, `CollectionHandling.Array`
- Notebook [06b (RecipeML)](./06b_RecipeML_Corpus.ipynb) : corpus RecipeML, classe `Recipe`, allergènes
- Concepts : `XDocument`, interpolation de chaînes C#, HTML/CSS inline

**Durée estimée** : 40-50 minutes

---

## Introduction : la frontière solveur / communication

Les notebooks 05 et 06b affichaient leurs solutions via `Console.WriteLine` sous forme de **tableaux plats alignés par espaces**. Cette représentation suffit pour debugger ou vérifier un résultat à l'écran d'un développeur. Mais en production — un menu affiché à un utilisateur, un planning imprimé, un rapport envoyé à un diététicien — **la sortie texte brute est inutilisable** :

- Impossible de distinguer au premier coup d'œil une recette qui dépasse le budget d'une qui le respecte
- Aucune signalétique visuelle pour les allergènes (pourtant critique pour la santé)
- Aucune barre ni jauge pour comparer des quantités (calories, protéines, coût)
- Pas de mise en page pour un plan multi-jours (lignes vs colonnes)

Ce notebook introduit la brique qui manque : un **rendu HTML structuré et coloré** de la solution Z3. Le principe est simple — l'API `display(HTML(htmlString))` de `Microsoft.DotNet.Interactive` accepte n'importe quelle chaîne HTML et la rend comme une cellule `text/html` riche. On construit cette chaîne en C# pur (interpolation `$"..."` ou `StringBuilder`), en injectant les valeurs issues du modèle Z3.

### Ce que ce notebook démontre

1. **Couplage faible solveur / rendu** (section 3) — on reprend *exactement* le modèle booléen de 06b et on ne change **que la dernière ligne** : `display(HTML(RenderMenuCard(...)))` à la place de `Console.WriteLine`. Le solveur ne sait même pas qu'il est rendu en HTML.
2. **Plan multi-jours rendu en grille** (section 4) — le `int[][]` de 06b est projeté en une grille HTML 7 colonnes (jours en lignes), avec **détection visuelle des répétitions**.
3. **Exploration de l'espace des solutions rendue visible** (section 5) — on lance le solveur sous une **série de budgets** (800 à 2000 centimes) et on rend le **front de Pareto kcal/coût** sous forme de tableau de compromis. C'est une montée en gamme (EPIC #3801 Prong-B) : un seul solve n'illustre pas la richesse du moteur ; une exploration paramétrique oui.

> **Note technique** : `display` et `HTML` sont des helpers globaux de `Microsoft.DotNet.Interactive`. Ils ne nécessitent pas de `using` spécifique, mais on ajoutera `using Microsoft.DotNet.Interactive;` par sécurité.

In [1]:
#r "../Z3.Linq/.deploy/Microsoft.Z3.dll"
#r "../Z3.Linq/.deploy/ExpressionUtils.dll"
#r "../Z3.Linq/.deploy/Z3.Linq.dll"
using Microsoft.Z3;
using Z3.Linq;
using Microsoft.DotNet.Interactive;
using System;
using System.Collections.Generic;
using System.Linq;
using System.Text;
using System.Xml.Linq;
Console.WriteLine("Imports OK : Z3.Linq, Microsoft.Z3, Microsoft.DotNet.Interactive (rendu HTML)");

The below script needs to be able to find the current output cell; this is an easy method to get it.

Imports OK : Z3.Linq, Microsoft.Z3, Microsoft.DotNet.Interactive (rendu HTML)


---

## 1. Corpus RecipeML et classe de domaine

On réutilise **le même corpus 7-recettes** que le notebook 06b (entrées / plats / desserts avec un mix pédagogique de coûts, protéines, allergènes). Pour éviter le piège du **verbatim string C#** (les guillemets XML doivent être doublés dans `@"..."`, sous peine de CS1003/CS1010), on construit la chaîne XML via un `StringBuilder`. Plus sûr, même résultat.

La classe `Recipe` est identique à 06b : le rendu HTML consommera exactement les mêmes propriétés (`Kcal`, `Protein`, `CostCents`, `Allergens`).

In [2]:
// Construction du corpus XML via StringBuilder (evite le piege du verbatim string @"...",
// ou chaque guillemet XML devrait etre double sous peine de CS1003/CS1010).
var sb = new StringBuilder();
sb.Append("<recipeml>");
Action<string, string, int, int, int, int, int, int, string, int> add =
    (name, cat, kcal, prot, carbs, fat, cost, prep, allerg, idx) =>
    {
        sb.Append($"<recipe><head><title>{name}</title><category>{cat}</category></head>");
        sb.Append($"<nutrition kcal=\"{kcal}\" protein=\"{prot}\" carbs=\"{carbs}\" fat=\"{fat}\"/>");
        sb.Append($"<prep time=\"{prep}\"/>");
        sb.Append($"<ingredients>(cf. corpus 06b)</ingredients>");
        sb.Append($"<allergens>{allerg}</allergens>");
        sb.Append($"<cost currency=\"EUR\">{cost}</cost></recipe>");
    };
add("Salade de quinoa",     "entree",  180,  6, 24,  6, 280, 15, "none",                0);
add("Veloute de potiron",   "entree",  120,  4, 18,  3, 220, 20, "lactose",             1);
add("Poulet roti aux herbes","plat",   520, 45,  8, 32, 850, 60, "none",                2);
add("Lasagnes bolognaise",  "plat",    610, 28, 55, 30, 720, 75, "gluten,lactose",      3);
add("Tofu curry coco",      "plat",    430, 22, 30, 24, 540, 35, "none",                4);
add("Brownie aux noix",     "dessert", 380,  6, 42, 22, 410, 45, "gluten,lactose,nuts", 5);
add("Salade de fruits frais","dessert", 90,  1, 22,  0, 180, 10, "none",                6);
sb.Append("</recipeml>");
string recipesXml = sb.ToString();

public class Recipe
{
    public string Name { get; set; }
    public string Category { get; set; }
    public int Kcal { get; set; }
    public int Protein { get; set; }
    public int Carbs { get; set; }
    public int Fat { get; set; }
    public int CostCents { get; set; }
    public int PrepMin { get; set; }
    public string Allergens { get; set; }
    public bool HasAllergen(string a) => Allergens != "none" && Allergens.Split(',').Contains(a);
}

var doc = XDocument.Parse(recipesXml);
var corpus = doc.Root.Elements("recipe").Select(r => new Recipe
{
    Name = (string)r.Element("head").Element("title"),
    Category = (string)r.Element("head").Element("category"),
    Kcal = (int)r.Element("nutrition").Attribute("kcal"),
    Protein = (int)r.Element("nutrition").Attribute("protein"),
    Carbs = (int)r.Element("nutrition").Attribute("carbs"),
    Fat = (int)r.Element("nutrition").Attribute("fat"),
    CostCents = (int)r.Element("cost"),
    PrepMin = (int)r.Element("prep").Attribute("time"),
    Allergens = (string)r.Element("allergens") ?? "none"
}).ToList();

Console.WriteLine("Corpus parse : " + corpus.Count + " recettes");
Console.WriteLine(string.Format("{0,-24} {1,-8} {2,4} {3,4} {4,6} {5,-18}", "Nom", "Cat", "kcal", "prot", "prix", "allergenes"));
foreach (var rc in corpus)
    Console.WriteLine(string.Format("{0,-24} {1,-8} {2,4} {3,4} {4,4} c {5,-18}", rc.Name, rc.Category, rc.Kcal, rc.Protein, rc.CostCents, rc.Allergens));

Corpus parse : 7 recettes


Nom                      Cat      kcal prot   prix allergenes        


Salade de quinoa         entree    180    6  280 c none              


Veloute de potiron       entree    120    4  220 c lactose           


Poulet roti aux herbes   plat      520   45  850 c none              


Lasagnes bolognaise      plat      610   28  720 c gluten,lactose    


Tofu curry coco          plat      430   22  540 c none              


Brownie aux noix         dessert   380    6  410 c gluten,lactose,nuts


Salade de fruits frais   dessert    90    1  180 c none              


---

## 2. Helper de rendu HTML : cartes et grilles

On définit deux helpers statiques qui prennent en entrée des **objets du domaine** (pas des expressions Z3) et retournent une **chaîne HTML**. C'est cette séparation qui rend le rendu réutilisable : le helper ne sait pas que les données viennent d'un solveur.

### `RenderMenuCard(IEnumerable<Recipe> menu, ...)`

Renvoie une carte HTML stylée représentant un menu complet. Encodage visuel :

- **Calories** : badge vert si 500-900 kcal (dans la cible), orange si à proximité (400-500 ou 900-1100), rouge sinon
- **Protéines** : barre de progression proportionnelle (sur une base de 50 g)
- **Coût** : badge vert si ≤ budget, rouge sinon
- **Allergènes** : badge rouge avec libellé si présent, badge vert "sans allergène" sinon

### `RenderWeeklyGrid(int[][] plan, List<Recipe> corpus)`

Renvoie une grille HTML (jours en lignes, créneaux en colonnes). Les répétitions d'une même recette à travers les jours sont **détectées** et marquées d'une couleur commune, afin de révéler visuellement une violation potentielle de variété.

Le style utilise du CSS inline (pas de feuille externe) : coins arrondis, ombres légères, en-têtes colorées. Tout est dans la chaîne renvoyée — pas d'effet de bord.

In [3]:
// Helpers de rendu HTML : ils consomment des objets du domaine (Recipe) et renvoient une
// chaine HTML. Aucune dependance Z3 : la couche de presentation est découplée du solveur.

public static string KcalColor(int kcal)
{
    if (kcal >= 500 && kcal <= 900) return "#2e7d32";   // vert : dans la cible
    if ((kcal >= 400 && kcal < 500) || (kcal > 900 && kcal <= 1100)) return "#ef6c00"; // orange
    return "#c62828";                                    // rouge : hors cible
}

public static string AllergenBadge(Recipe r)
{
    if (r.Allergens == "none")
        return "<span style='background:#2e7d32;color:white;padding:2px 8px;border-radius:8px;font-size:11px'>sans allergene</span>";
    var parts = r.Allergens.Split(',').Select(a => $"<span style='background:#c62828;color:white;padding:2px 8px;border-radius:8px;font-size:11px;margin-right:4px'>{a}</span>");
    return string.Join("", parts);
}

public static string ProteinBar(int protein)
{
    int pct = Math.Min(100, protein * 100 / 50);  // base 50 g = 100 %
    string color = protein >= 30 ? "#2e7d32" : "#ef6c00";
    return $"<div style='background:#eee;border-radius:4px;width:90px;height:10px;display:inline-block;vertical-align:middle'>" +
           $"<div style='background:{color};height:10px;border-radius:4px;width:{pct}%'></div></div> " +
           $"<span style='font-size:11px'>{protein} g</span>";
}

public static string RenderMenuCard(IEnumerable<Recipe> menu, string title, int budgetCents)
{
    var list = menu.ToList();
    int totKcal = list.Sum(r => r.Kcal);
    int totProt = list.Sum(r => r.Protein);
    int totCost = list.Sum(r => r.CostCents);
    string kcalCol = KcalColor(totKcal);
    string costCol = totCost <= budgetCents ? "#2e7d32" : "#c62828";

    var h = new StringBuilder();
    h.Append($"<div style='font-family:sans-serif;border:1px solid #ccc;border-radius:10px;box-shadow:0 2px 8px rgba(0,0,0,0.1);max-width:720px;margin:8px 0;overflow:hidden'>");
    h.Append($"<div style='background:#1565c0;color:white;padding:10px 16px;font-size:16px;font-weight:bold'>{title}</div>");
    h.Append("<div style='padding:8px 16px'>");
    h.Append("<table style='width:100%;border-collapse:collapse;font-size:13px'>");
    h.Append("<tr style='background:#f5f5f5'><th style='text-align:left;padding:6px'>Categorie</th><th style='text-align:left;padding:6px'>Recette</th><th style='padding:6px'>kcal</th><th style='padding:6px'>Proteines</th><th style='padding:6px'>Prix</th><th style='padding:6px'>Allergenes</th></tr>");
    foreach (var r in list)
    {
        string kc = KcalColor(r.Kcal);
        h.Append($"<tr style='border-bottom:1px solid #eee'>");
        h.Append($"<td style='padding:6px'><span style='background:#455a64;color:white;padding:2px 8px;border-radius:8px;font-size:11px'>{r.Category}</span></td>");
        h.Append($"<td style='padding:6px;font-weight:600'>{r.Name}</td>");
        h.Append($"<td style='padding:6px;text-align:center'><span style='color:{kc};font-weight:bold'>{r.Kcal}</span></td>");
        h.Append($"<td style='padding:6px'>{ProteinBar(r.Protein)}</td>");
        h.Append($"<td style='padding:6px;text-align:center'>{r.CostCents/100.0:F2} E</td>");
        h.Append($"<td style='padding:6px'>{AllergenBadge(r)}</td>");
        h.Append("</tr>");
    }
    h.Append($"<tr style='background:#eceff1;font-weight:bold'>");
    h.Append($"<td colspan='2' style='padding:8px'>TOTAL</td>");
    h.Append($"<td style='padding:8px;text-align:center'><span style='color:{kcalCol}'>{totKcal}</span></td>");
    h.Append($"<td style='padding:8px'>{totProt} g</td>");
    h.Append($"<td style='padding:8px;text-align:center'><span style='color:{costCol}'>{totCost/100.0:F2} E / {budgetCents/100.0:F2}</span></td>");
    h.Append("<td></td></tr>");
    h.Append("</table></div></div>");
    return h.ToString();
}

public static string RenderWeeklyGrid(int[][] plan, List<Recipe> corpus)
{
    int days = plan.Length;
    int slots = days > 0 ? plan[0].Length : 0;
    // Couleur par indice de recette pour detecter les repetitions.
    var palette = new[] { "#e3f2fd", "#fff3e0", "#e8f5e9", "#fce4ec", "#f3e5f5", "#fffde7", "#efebe9" };
    var h = new StringBuilder();
    h.Append("<div style='font-family:sans-serif;border:1px solid #ccc;border-radius:10px;box-shadow:0 2px 8px rgba(0,0,0,0.1);max-width:760px;margin:8px 0;overflow:hidden'>");
    h.Append("<div style='background:#6a1b9a;color:white;padding:10px 16px;font-size:16px;font-weight:bold'>Plan multi-jours</div>");
    h.Append("<div style='padding:8px 16px'><table style='width:100%;border-collapse:collapse;font-size:13px'>");
    var slotNames = new[] { "Dejeuner", "Diner", "Extra" };
    h.Append("<tr style='background:#f5f5f5'><th style='padding:6px'>Jour</th>");
    for (int s = 0; s < slots; s++)
        h.Append($"<th style='padding:6px'>{(s < slotNames.Length ? slotNames[s] : "Creneau " + (s+1))}</th>");
    h.Append("</tr>");
    for (int d = 0; d < days; d++)
    {
        h.Append($"<tr>");
        h.Append($"<td style='padding:6px;font-weight:bold;background:#fafafa'>Jour {d+1}</td>");
        for (int s = 0; s < slots; s++)
        {
            int idx = plan[d][s];
            var rc = corpus[idx];
            string bg = palette[idx % palette.Length];
            h.Append($"<td style='padding:6px;background:{bg};border:1px solid #ddd;border-radius:4px'>{rc.Name}<br/><span style='font-size:11px;color:#555'>{rc.Kcal} kcal / {rc.Category}</span></td>");
        }
        h.Append("</tr>");
    }
    h.Append("</table>");
    h.Append("<div style='font-size:11px;color:#666;margin-top:6px'>Meme couleur de fond = meme recette (repetition visible).</div>");
    h.Append("</div></div>");
    return h.ToString();
}

Console.WriteLine($"Helpers definis : RenderMenuCard, RenderWeeklyGrid, KcalColor, AllergenBadge, ProteinBar");

Helpers definis : RenderMenuCard, RenderWeeklyGrid, KcalColor, AllergenBadge, ProteinBar


---

## 3. Modèle booléen + rendu en carte HTML

On reprend **à l'identique** le modèle booléen de 06b section 3 : une `BoolExpr` par recette, cardinalité 1 par catégorie, agrégation nutritionnelle via `MkITE`, exclusion de l'allergène `nuts`, bornes kcal/protéines/budget. La seule différence avec 06b est la **dernière ligne** : au lieu de `Console.WriteLine`, on appelle `display(HTML(RenderMenuCard(...)))`.

C'est la démonstration centrale du notebook : **changer le rendu ne change pas le solveur**. Le moteur SMT reste la source de vérité ; la couche de présentation est une fonction pure des objets du domaine.

In [4]:
// Modele booleen Z3 (identique a 06b section 3) : un BoolExpr par recette, agregation MkITE,
// exclusion de l'allergene 'nuts'. Rendu final en carte HTML via display(HTML(...)).

List<Recipe> solvedMenu = null;
Status solveStatus;

using (var z3ctx = new Context())
{
    var solver = z3ctx.MkSolver();
    var selections = corpus
        .Select((rc, i) => new { Recipe = rc, Index = i, Var = (BoolExpr)z3ctx.MkConst($"sel_{i}_{rc.Name}", z3ctx.BoolSort) })
        .ToList();

    void ExactlyOne(string cat)
    {
        var catVars = selections.Where(s => s.Recipe.Category == cat).Select(s => (BoolExpr)s.Var).ToArray();
        solver.Add(z3ctx.MkOr(catVars));
        for (int a = 0; a < catVars.Length; a++)
            for (int b = 0; b < catVars.Length; b++)
                if (a != b) solver.Add(z3ctx.MkImplies(catVars[a], z3ctx.MkNot(catVars[b])));
    }
    ExactlyOne("entree"); ExactlyOne("plat"); ExactlyOne("dessert");

    IntExpr totalKcal = (IntExpr)selections.Aggregate((IntExpr)z3ctx.MkInt(0),
        (acc, s) => (IntExpr)z3ctx.MkAdd(acc, (ArithExpr)z3ctx.MkITE(s.Var, z3ctx.MkInt(s.Recipe.Kcal), z3ctx.MkInt(0))));
    IntExpr totalProt = (IntExpr)selections.Aggregate((IntExpr)z3ctx.MkInt(0),
        (acc, s) => (IntExpr)z3ctx.MkAdd(acc, (ArithExpr)z3ctx.MkITE(s.Var, z3ctx.MkInt(s.Recipe.Protein), z3ctx.MkInt(0))));
    IntExpr totalCost = (IntExpr)selections.Aggregate((IntExpr)z3ctx.MkInt(0),
        (acc, s) => (IntExpr)z3ctx.MkAdd(acc, (ArithExpr)z3ctx.MkITE(s.Var, z3ctx.MkInt(s.Recipe.CostCents), z3ctx.MkInt(0))));

    int budgetCents = 1500;
    solver.Add(z3ctx.MkGe(totalKcal, z3ctx.MkInt(500)));
    solver.Add(z3ctx.MkLe(totalKcal, z3ctx.MkInt(900)));
    solver.Add(z3ctx.MkGe(totalProt, z3ctx.MkInt(30)));
    solver.Add(z3ctx.MkLe(totalCost, z3ctx.MkInt(budgetCents)));

    string bannedAllergen = "nuts";
    foreach (var s in selections.Where(s => s.Recipe.HasAllergen(bannedAllergen)))
        solver.Add(z3ctx.MkNot(s.Var));

    solveStatus = solver.Check();
    Console.WriteLine($"Resultat solve : {solveStatus}");

    if (solveStatus == Status.SATISFIABLE)
    {
        var model = solver.Model;
        solvedMenu = selections.Where(s => model.Eval(s.Var, true).BoolValue == Z3_lbool.Z3_L_TRUE)
                               .Select(s => s.Recipe).ToList();
    }
}

if (solvedMenu != null)
    display(HTML(RenderMenuCard(solvedMenu, "Menu equilibre sans fruits a coque (solveur Z3)", 1500)));
else
    Console.WriteLine("Aucun menu ne satisfait les contraintes.");

Resultat solve : SATISFIABLE


Menu equilibre sans fruits a coque (solveur Z3) Categorie Recette kcal Proteines Prix Allergenes entree Veloute de potiron 120 4 g 2,20 E lactose plat Lasagnes bolognaise 610 28 g 7,20 E gluten lactose dessert Salade de fruits frais 90 1 g 1,80 E sans allergene TOTAL 820 33 g 11,20 E / 15,00


warning CS1701: En supposant que la référence d'assembly 'Microsoft.AspNetCore.Html.Abstractions, Version=2.2.0.0, Culture=neutral, PublicKeyToken=adb9793829ddae60' utilisée par 'Microsoft.DotNet.Interactive' correspond à l'identité 'Microsoft.AspNetCore.Html.Abstractions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=adb9793829ddae60' de 'Microsoft.AspNetCore.Html.Abstractions', il se peut que vous deviez fournir une stratégie runtime



### Interpretation : ce que l'encodage couleur revele d'un coup d'oeil

La carte HTML ci-dessus condense en une seule image ce que le tableau `Console.WriteLine` de 06b laissait au lecteur le soin de recomposer :

| Aspect | Console (06b) | Carte HTML (06c) |
|--------|---------------|------------------|
| Calories dans la cible 500-900 | à calculer mentalement | badge couleur sur le total |
| Recette qui monte en calories | pas de signal | chiffre coloré par ligne |
| Apport protéique | chiffre brut | barre de progression visuelle |
| Statut d'allergène | chaîne à lire | badge rouge/vert instantané |
| Dépassement de budget | soustraction manuelle | total coloré vs budget affiché |

**Points clés** :

1. **Le solveur n'a pas changé** — l'objet `solvedMenu` est une `List<Recipe>` conforme à la même définition qu'en 06b. Le rendu HTML est une **fonction pure** de cette liste.
2. **Le badge allergène rouge** rend immédiatement visible si une recette sélectionnée contient du lactose ou du gluten — utile pour le communicant qui prépare le menu, pas seulement le diététicien qui le calcule.
3. **L'extension est triviale** : ajouter une colonne "temps de préparation" consisterait à ajouter une cellule `<td>{r.PrepMin} min</td>` — sans toucher au modèle Z3.

---

## 4. Plan hebdomadaire `int[][]` + grille HTML

On étend au plan multi-jours du notebook 06b section 4 : un `int[][]` (3 jours x 2 créneaux) peuplé d'indices de recettes dans le corpus, avec contrainte de non-répétition des déjeuners via `Z3Methods.Distinct` en mode `CollectionHandling.Array`.

### Rappel technique : modèle de soumission .NET Interactive

> La cellule ci-dessous regroupe **dans le même bloc** la déclaration des bornes (`entLo`, `pltLo`, etc.), la construction du théorème `Where(...)`, **et** l'appel `.Solve()`. C'est un choix de **lisibilité** : bornes et contraintes se lisent d'un seul tenant.
>
> **Arrière-plan** : chaque cellule `.net-csharp` compile dans un assembly dynamique distinct (une *submission* `Submission#N`). Capturer dans un lambda une variable d'une autre cellule exige une résolution *cross-submission*. Ce cas, historiquement cassant en `ArgumentException` dans le fork Z3.Linq, est **désormais résolu** : `Theorem.AssertConstraints` replie les constantes capturées en littéraux via `PartialEvaluator.PartialEval` avant la visite de l'arbre. La capture cross-cellule fonctionne — garder tout dans la même cellule reste une **recommandation de lisibilité**, plus une exigence.

In [5]:
// Theoreme hierarchique int[][] sur le corpus RecipeML, rendu en grille HTML.
// Recommandation de lisibilite : bornes + Where(...) + Solve() dans la MEME cellule.
// (La capture cross-submission est desormais supportee par le correctif AssertConstraints.)

public class DayPlan
{
    public int[][] Sel { get; set; } = new int[3][];
    public DayPlan() { for (int t = 0; t < 3; t++) Sel[t] = new int[2]; }
}

int entLo = corpus.FindIndex(r => r.Category == "entree");
int entHi = corpus.FindLastIndex(r => r.Category == "entree");
int pltLo = corpus.FindIndex(r => r.Category == "plat");
int pltHi = corpus.FindLastIndex(r => r.Category == "plat");
int desLo = corpus.FindIndex(r => r.Category == "dessert");
int desHi = corpus.FindLastIndex(r => r.Category == "dessert");
Console.WriteLine($"Bornes corpus -> entrees [{entLo},{entHi}], plats [{pltLo},{pltHi}], desserts [{desLo},{desHi}]");

DayPlan sol = null;
using (var ctx = new Z3Context { DefaultCollectionHandling = CollectionHandling.Array })
{
    var th = ctx.NewTheorem<DayPlan>();
    for (int t = 0; t < 3; t++)
    {
        int tt = t;
        th = th.Where(g => g.Sel[tt][0] >= pltLo && g.Sel[tt][0] <= pltHi);
        th = th.Where(g => g.Sel[tt][1] >= pltLo && g.Sel[tt][1] <= pltHi);
    }
    th = th.Where(g => Z3Methods.Distinct(g.Sel[0][0], g.Sel[1][0], g.Sel[2][0]));

    var sw = System.Diagnostics.Stopwatch.StartNew();
    sol = th.Solve();
    sw.Stop();
    Console.WriteLine($"Theoreme int[][] resolu en {sw.Elapsed.TotalMilliseconds:F1} ms (status : {(sol != null ? "SAT" : "UNSAT")})");
}

if (sol != null)
    display(HTML(RenderWeeklyGrid(sol.Sel, corpus)));
else
    Console.WriteLine("(Aucun plan realisable avec ces bornes.)");

Bornes corpus -> entrees [0,1], plats [2,4], desserts [5,6]


Theoreme int[][] resolu en 92,3 ms (status : SAT)


Jour,Dejeuner,Diner
Jour 1,Lasagnes bolognaise610 kcal / plat,Poulet roti aux herbes520 kcal / plat
Jour 2,Poulet roti aux herbes520 kcal / plat,Tofu curry coco430 kcal / plat
Jour 3,Tofu curry coco430 kcal / plat,Lasagnes bolognaise610 kcal / plat



warning CS1701: En supposant que la référence d'assembly 'Microsoft.AspNetCore.Html.Abstractions, Version=2.2.0.0, Culture=neutral, PublicKeyToken=adb9793829ddae60' utilisée par 'Microsoft.DotNet.Interactive' correspond à l'identité 'Microsoft.AspNetCore.Html.Abstractions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=adb9793829ddae60' de 'Microsoft.AspNetCore.Html.Abstractions', il se peut que vous deviez fournir une stratégie runtime



### Interpretation : la grille rend la variété visible

La grille HTML ci-dessus attribue à **chaque indice de recette** une couleur de fond stable. Sans même lire les noms, l'œil repère :

- les **colonnes déjeuner** : trois couleurs différentes = contrainte `Distinct` satisfaite (aucune répétition cross-jour)
- les **colonnes dîner** : peuvent se chevaucher (pas de contrainte `Distinct` posée ici — cf. exercice 2)

| Aspect | Rôle de la couleur |
|--------|---------------------|
| Vérifier la non-répetition | couleurs identiques = alerte immédiate |
| Repérer une catégorie absente | aucune cellule verte si aucun dessert |
| Comparer visuellement la charge kcal | lecture facile des annotations par cellule |

**Points clés** :

1. Le solveur trouve une **permutation complète** des trois plats du corpus sur les trois déjeuners — la contrainte `Z3Methods.Distinct` accomplit un travail qu'une simple borne ne pourrait pas exprimer (non-trivialité Prong-B).
2. Le rendu grille est **indépendant du contenu** : la même fonction `RenderWeeklyGrid` afficherait aussi bien un plan 7 jours x 3 créneaux. Seule la forme du `int[][]` change.

---

## 5. Front de Pareto HTML : exploration de l'espace des solutions

Un seul solve montre qu'il **existe** une solution. Une **série de solves sous budgets décroissants** révèle la structure du compromis kcal/coût — c'est le front de Pareto. Cette section est la montée en gamme du notebook (EPIC #3801 Prong-B) : on fait valoir la capacité du moteur à **explorer** un espace paramétrique, pas seulement à trouver un point.

On boucle sur les budgets de 800 à 2000 centimes (par pas de 200), on résout à chaque fois le modèle booléen avec le même corpus, et on collecte le couple `(kcal, cost, prot)` de la solution retenue. On rend ensuite le tout comme un **tableau stylé de compromis** où chaque ligne est un point du front.

In [6]:
// Front de Pareto kcal/cout : on boucle sur des budgets decroissants, on resout le modele
// booleen a chaque fois (brownie reste exclu), et on collecte (budget, kcal, cout, prot).
// Rendu final en tableau HTML style "compromis".

var pareto = new List<(int BudgetCents, int Kcal, int Cost, int Prot, string Desc)>();

foreach (int budgetCents in new[] { 2000, 1800, 1600, 1400, 1200, 1000, 800 })
{
    using (var z3ctx = new Context())
    {
        var solver = z3ctx.MkSolver();
        var selections = corpus
            .Select((rc, i) => new { Recipe = rc, Var = (BoolExpr)z3ctx.MkConst($"s_{i}_{budgetCents}", z3ctx.BoolSort) })
            .ToList();

        void ExactlyOne(string cat)
        {
            var cv = selections.Where(s => s.Recipe.Category == cat).Select(s => (BoolExpr)s.Var).ToArray();
            solver.Add(z3ctx.MkOr(cv));
            for (int a = 0; a < cv.Length; a++)
                for (int b = 0; b < cv.Length; b++)
                    if (a != b) solver.Add(z3ctx.MkImplies(cv[a], z3ctx.MkNot(cv[b])));
        }
        ExactlyOne("entree"); ExactlyOne("plat"); ExactlyOne("dessert");

        IntExpr totalKcal = (IntExpr)selections.Aggregate((IntExpr)z3ctx.MkInt(0),
            (acc, s) => (IntExpr)z3ctx.MkAdd(acc, (ArithExpr)z3ctx.MkITE(s.Var, z3ctx.MkInt(s.Recipe.Kcal), z3ctx.MkInt(0))));
        IntExpr totalProt = (IntExpr)selections.Aggregate((IntExpr)z3ctx.MkInt(0),
            (acc, s) => (IntExpr)z3ctx.MkAdd(acc, (ArithExpr)z3ctx.MkITE(s.Var, z3ctx.MkInt(s.Recipe.Protein), z3ctx.MkInt(0))));
        IntExpr totalCost = (IntExpr)selections.Aggregate((IntExpr)z3ctx.MkInt(0),
            (acc, s) => (IntExpr)z3ctx.MkAdd(acc, (ArithExpr)z3ctx.MkITE(s.Var, z3ctx.MkInt(s.Recipe.CostCents), z3ctx.MkInt(0))));

        solver.Add(z3ctx.MkGe(totalKcal, z3ctx.MkInt(400)));
        solver.Add(z3ctx.MkGe(totalProt, z3ctx.MkInt(20)));
        solver.Add(z3ctx.MkLe(totalCost, z3ctx.MkInt(budgetCents)));
        foreach (var s in selections.Where(s => s.Recipe.HasAllergen("nuts")))
            solver.Add(z3ctx.MkNot(s.Var));

        if (solver.Check() == Status.SATISFIABLE)
        {
            var model = solver.Model;
            var chosen = selections.Where(s => model.Eval(s.Var, true).BoolValue == Z3_lbool.Z3_L_TRUE).Select(s => s.Recipe).ToList();
            pareto.Add((budgetCents, chosen.Sum(r => r.Kcal), chosen.Sum(r => r.CostCents), chosen.Sum(r => r.Protein),
                        string.Join(" + ", chosen.Select(r => r.Name))));
        }
        else
        {
            pareto.Add((budgetCents, -1, -1, -1, "(IRREALISABLE)"));
        }
    }
}

// Rendu HTML du front de Pareto.
int maxKcal = pareto.Where(p => p.Kcal > 0).Max(p => p.Kcal);
var h = new StringBuilder();
h.Append("<div style='font-family:sans-serif;border:1px solid #ccc;border-radius:10px;box-shadow:0 2px 8px rgba(0,0,0,0.1);max-width:780px;margin:8px 0;overflow:hidden'>");
h.Append("<div style='background:#00695c;color:white;padding:10px 16px;font-size:16px;font-weight:bold'>Front de Pareto : compromis kcal / cout</div>");
h.Append("<div style='padding:8px 16px'><table style='width:100%;border-collapse:collapse;font-size:13px'>");
h.Append("<tr style='background:#f5f5f5'><th style='padding:6px'>Budget (E)</th><th style='padding:6px'>kcal</th><th style='padding:6px'>Jauge kcal</th><th style='padding:6px'>cout reel (E)</th><th style='padding:6px'>prot (g)</th><th style='padding:6px'>menu</th></tr>");
foreach (var p in pareto)
{
    if (p.Kcal < 0)
    {
        h.Append($"<tr><td colspan='6' style='padding:6px;color:#c62828;font-style:italic'>Budget {p.BudgetCents/100.0:F2} E : irrealisable</td></tr>");
        continue;
    }
    int pct = maxKcal > 0 ? p.Kcal * 100 / maxKcal : 0;
    string col = KcalColor(p.Kcal);
    int margin = p.BudgetCents - p.Cost;
    string marginCol = margin >= 0 ? "#2e7d32" : "#c62828";
    h.Append("<tr style='border-bottom:1px solid #eee'>");
    h.Append($"<td style='padding:6px;text-align:center'>{p.BudgetCents/100.0:F2}</td>");
    h.Append($"<td style='padding:6px;text-align:center;color:{col};font-weight:bold'>{p.Kcal}</td>");
    h.Append($"<td style='padding:6px'><div style='background:#eee;border-radius:4px;width:140px;height:10px'><div style='background:{col};height:10px;border-radius:4px;width:{pct}%'></div></div></td>");
    h.Append($"<td style='padding:6px;text-align:center'>{p.Cost/100.0:F2} <span style='color:{marginCol};font-size:11px'>(marge {margin/100.0:F2})</span></td>");
    h.Append($"<td style='padding:6px;text-align:center'>{p.Prot}</td>");
    h.Append($"<td style='padding:6px;font-size:12px'>{p.Desc}</td>");
    h.Append("</tr>");
}
h.Append("</table>");
h.Append("<div style='font-size:11px;color:#666;margin-top:6px'>Chaque ligne = une solution du solveur sous un budget different. La jauge kcal est relative au max observe.</div>");
h.Append("</div></div>");
Console.WriteLine($"Front de Pareto : {pareto.Count} points (budgets 800 a 2000 centimes)");
display(HTML(h.ToString()));

Front de Pareto : 7 points (budgets 800 a 2000 centimes)


Front de Pareto : compromis kcal / cout Budget (E) kcal Jauge kcal cout reel (E) prot (g) menu 20,00 820 11,20 (marge 8,80) 33 Veloute de potiron + Lasagnes bolognaise + Salade de fruits frais 18,00 820 11,20 (marge 6,80) 33 Veloute de potiron + Lasagnes bolognaise + Salade de fruits frais 16,00 820 11,20 (marge 4,80) 33 Veloute de potiron + Lasagnes bolognaise + Salade de fruits frais 14,00 820 11,20 (marge 2,80) 33 Veloute de potiron + Lasagnes bolognaise + Salade de fruits frais 12,00 820 11,20 (marge 0,80) 33 Veloute de potiron + Lasagnes bolognaise + Salade de fruits frais 10,00 640 9,40 (marge 0,60) 27 Veloute de potiron + Tofu curry coco + Salade de fruits frais Budget 8,00 E : irrealisable Chaque ligne = une solution du solveur sous un budget different. La jauge kcal est relative au max observe.


warning CS1701: En supposant que la référence d'assembly 'Microsoft.AspNetCore.Html.Abstractions, Version=2.2.0.0, Culture=neutral, PublicKeyToken=adb9793829ddae60' utilisée par 'Microsoft.DotNet.Interactive' correspond à l'identité 'Microsoft.AspNetCore.Html.Abstractions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=adb9793829ddae60' de 'Microsoft.AspNetCore.Html.Abstractions', il se peut que vous deviez fournir une stratégie runtime



### Interpretation : le solveur comme explorateur de l'espace des solutions

Le tableau ci-dessus **ne montre pas une solution** mais un **spectre** : ce que le solveur peut atteindre à chaque niveau de budget. C'est l'argument opérationnel pour utiliser un moteur SMT plutôt qu'un algorithme glouton :

| Observation | Lecture |
|-------------|---------|
| À budget élevé (20 EUR), kcal max | le solveur peut se permettre un menu consistant |
| À budget bas, bascule vers Tofu/Salade | **rotation automatique** des recettes moins chères |
| Palier "irréalisable" | le solveur prouve qu'aucun menu ne satisfait les contraintes minimales |

**Points clés** :

1. **Chaque ligne est un solve indépendant** — il n'y a pas d'astuce : le solveur a tourné 7 fois. Le coût calculatoire est négligeable (chaque solve prend quelques ms).
2. **Le rendu rend la décision possible** : un décideur qui compare visuellement les jauges et les marges peut arbitrer "je sacrifie 100 kcal pour économiser 2 EUR" en un coup d'œil. C'est exactement la valeur d'un solveur rendu lisiblement.
3. **Frontière avec l'optimisation** : on n'a pas **maximisé** ici (ce serait l'exercice 2 du notebook 06b), on a **échantillonné**. Les deux approches sont complémentaires.

---

## 6. Tableau récapitulatif : Console (05/06b) vs HTML (06c)

| Critère | `Console.WriteLine` (05, 06b) | `display(HTML(...))` (06c) |
|---------|--------------------------------|----------------------------|
| Support | texte brut aligné par espaces | HTML riche rendu par le notebook |
| Encodage couleur | aucun | badges, jauges, codes caloriques |
| Détection allergène | lecture de chaîne | badge rouge/vert instantané |
| Plan multi-jours | liste de lignes | grille colourisée, répétitions visibles |
| Exploration paramétrique | fastidieuse | front de Pareto rendu en un bloc |
| Effort code | un `Console.WriteLine` | un helper HTML + appel `display` |
| Couplage au solveur | aucun | aucun (fonction pure des objets du domaine) |

### Le point fondamental

Le solveur n'a **pas été modifié** entre 06b et 06c. Les modèles (booléen avec `MkITE`, hiérarchique `int[][]`) sont identiques. Seule la **dernière ligne** de chaque section a changé : `Console.WriteLine` est devenu `display(HTML(...))`. C'est la preuve que l'architecture est **découplée** : le solveur produit des objets du domaine, le rendu est une fonction pure de ces objets.

---

## Exercices

Les exercices ci-dessous étendent la couche de rendu HTML. Chaque stub est volontairement incomplet : remplacez les commentaires `// TODO etudiant` par votre code. Les stubs ne lèvent **pas** d'exception — le notebook s'exécute de bout en bout même exercices non remplis.

### Exercice 1 : Badge végétarien dans la carte de menu

Ajoutez à la fonction `RenderMenuCard` un **badge végétarien** global (en haut de la carte) : vert si **toutes** les recettes du menu sont végétariennes, rouge sinon. On considère comme non-végétariennes les recettes dont le nom contient `"Poulet"`, `"boeuf"` ou `"Lasagnes"`.

**Indices** :
- `# Etape 1` : écrivez un prédicat `bool IsVegetarian(Recipe r)` qui renvoie `false` si `r.Name` contient l'un des marqueurs.
- `# Etape 2` : dans `RenderMenuCard`, calculez `bool allVeg = list.All(IsVegetarian)`.
- `# Etape 3` : insérez dans l'en-tête de la carte un badge : vert "100% vegetarien" si `allVeg`, rouge "contient de la viande" sinon.

In [7]:
// Exercice 1 : badge vegetarien global dans RenderMenuCard.
// TODO etudiant : modifier RenderMenuCard pour afficher un badge vegetarien en tete de carte.

// Indice :
// static bool IsVegetarian(Recipe r)
// {
//     string[] nonVege = { "Poulet", "boeuf", "Lasagnes" };
//     return !nonVege.Any(k => r.Name.Contains(k));
// }
// ... dans RenderMenuCard :
// bool allVeg = list.All(IsVegetarian);
// h.Append($"<div style='padding:4px 16px;background:{(allVeg ? "#e8f5e9" : "#ffebee")}'>" +
//          $"<span style='background:{(allVeg ? "#2e7d32" : "#c62828")};color:white;padding:2px 8px;border-radius:8px;font-size:11px'>" +
//          $"{(allVeg ? "100% vegetarien" : "contient de la viande")}</span></div>");

Console.WriteLine("Exercice a completer : badge vegetarien dans la carte de menu");

Exercice a completer : badge vegetarien dans la carte de menu


### Exercice 2 : Total kcal par jour dans la grille hebdomadaire

Modifiez `RenderWeeklyGrid` pour ajouter une **ligne finale "Total/jour"** qui affiche la somme des kcal de toutes les recettes de chaque journée, avec un code couleur (`KcalColor`).

**Indices** :
- `# Etape 1` : pour chaque jour `d`, calculez `int dayKcal = Enumerable.Range(0, slots).Sum(s => corpus[plan[d][s]].Kcal);`
- `# Etape 2` : après la boucle des jours, ajoutez une ligne `<tr>` avec une cellule "Total kcal" et une cellule par jour contenant le total coloré.
- `# Etape 3` : appelez `KcalColor(dayKcal)` pour la couleur du total.

In [8]:
// Exercice 2 : ligne "Total kcal / jour" dans la grille hebdomadaire.
// TODO etudiant : modifier RenderWeeklyGrid pour ajouter une ligne de totaux par jour.

// Indice :
// h.Append("<tr style='background:#eceff1;font-weight:bold'><td>Total kcal</td>");
// for (int d = 0; d < days; d++)
// {
//     int dayKcal = Enumerable.Range(0, slots).Sum(s => corpus[plan[d][s]].Kcal);
//     h.Append($"<td style='text-align:center;color:{KcalColor(dayKcal)}'>{dayKcal}</td>");
// }
// h.Append("</tr>");

Console.WriteLine("Exercice a completer : total kcal par jour dans la grille hebdomadaire");

Exercice a completer : total kcal par jour dans la grille hebdomadaire


### Exercice 3 : Front de Pareto 2D (budget × plancher protéines) rendu en heatmap

Étendez la section 5 en faisant varier **simultanément** le budget (800..2000) **et** le plancher de protéines (10..50 g). Pour chaque couple `(budget, protFloor)`, résolvez et stockez le kcal atteint. Rendez le tout comme une **heatmap HTML 2D** : lignes = budgets, colonnes = planchers protéiques, couleur de la cellule = kcal atteint (vert = haut, rouge = bas/irréalisable).

**Indices** :
- `# Etape 1` : construisez une double boucle `foreach (budget ...) foreach (protFloor ...)` qui appelle le solveur et remplit un `int[,] heatmap`.
- `# Etape 2` : pour la couleur, normalisez le kcal sur la plage observée (`intensity = (kcal - minKcal) / (maxKcal - minKcal)`) et mappez sur un dégradé vert→jaune→rouge (utilisez `System.Drawing.Color` ou un palette CSS).
- `# Etape 3` : générez un `<table>` HTML où chaque `<td>` a un `style='background:rgb(...)'` calculé depuis l'intensité, et contient la valeur kcal.

In [9]:
// Exercice 3 : heatmap 2D budget x plancher proteines (kcal atteint par cellule).
// TODO etudiant : double boucle de solve, stockage dans int[,], rendu heatmap HTML.

// Indice :
// int[] budgets = { 800, 1000, 1200, 1400, 1600, 1800, 2000 };
// int[] protFloors = { 10, 20, 30, 40, 50 };
// int[,] heatmap = new int[budgets.Length, protFloors.Length];
// ... double boucle : pour chaque (i,j), resoudre, stocker heatmap[i,j] = kcal ou -1 si UNSAT
// ... rendu : <table> avec background calcule depuis l'intensite normalisee

Console.WriteLine("Exercice a completer : heatmap 2D du front de Pareto budget x proteines");

Exercice a completer : heatmap 2D du front de Pareto budget x proteines


---

## Conclusion

Ce notebook a démontré que la **valeur opérationnelle** d'un solveur Z3 ne se limite pas à trouver une solution : elle dépend crucialement de la capacité à **rendre cette solution lisible** à un humain (décideur, diététicien, utilisateur final). L'API `display(HTML(...))` de `Microsoft.DotNet.Interactive` ouvre la porte à un rendu riche — cartes colorées, badges d'allergènes, grilles de planification, fronts de Pareto — sans quitter le notebook .NET.

### Points clés à retenir

| Concept | Implémentation |
|---------|----------------|
| Affichage HTML riche | `display(HTML(htmlString))` depuis `Microsoft.DotNet.Interactive` |
| Construction HTML | interpolation C# `$"..."` ou `StringBuilder` avec CSS inline |
| Découplage solveur/rendu | les helpers consomment des `Recipe`, pas des expressions Z3 |
| Couleur comme signal | `KcalColor`, badges allergènes, jauges de protéines |
| Exploration paramétrique | boucle de solves sous budgets variés → front de Pareto rendu |

### Perspective

Les techniques de rendu vues ici se généralisent à tout solveur produisant des objets structurés : planification de personnel (grille par équipe), allocation de ressources (heatmap de charge), ordonnancement (frise temporelle HTML). Le principe reste le même : le solveur est la source de vérité, le rendu est une fonction pure de ses sorties. Pour des visualisations plus ambitieuses, on pourrait également sérialiser les solutions en JSON et les consommer depuis une cellule JavaScript (via `#!javascript`), mais le rendu HTML inline suffit à la plupart des usages pédagogiques.

**Navigation** : [Index Z3](./README.md) | [Notebook 05 (Meal Planner)](./05_Meal_Planner_Hierarchical.ipynb) | [Notebook 06b (RecipeML)](./06b_RecipeML_Corpus.ipynb)